# Apriori — Association Rule Mining

- Apriori: finds frequent itemsets and association rules in transactional data
- Key metrics: Support (how often), Confidence (how likely given antecedent), Lift (strength of association)
- Dataset: Online_Retail.csv (UK e-commerce transactions — finding products frequently bought together)[https://www.kaggle.com/datasets/ulrikthygepedersen/online-retail-dataset](https://)

In [3]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [4]:
df = pd.read_csv('/content/online_retail[1].csv')
print(df)

       InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365     71053                  WHITE METAL LANTERN         6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
...          ...       ...                                  ...       ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
541906    581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
541907    581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
541908    581587     22138        BAKING SET 9 PIECE RETROSPOT          3   

                InvoiceDate  UnitPrice  CustomerID         Country  
0     

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [6]:
# Drop rows with missing CustomerID or Description
df = df.dropna(subset=["CustomerID", "Description"])

# Keep only positive quantities (no returns/cancellations)
df = df[df["Quantity"] > 0]

print("Shape after cleaning:", df.shape)
print(df.head())

Shape after cleaning: (397924, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [7]:
# Group by invoice and product — 1 if product was in that invoice, 0 otherwise
basket = df.groupby(["InvoiceNo", "Description"])["Quantity"].sum().unstack().fillna(0)

# Convert quantities to binary (bought or not)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)
print("Basket shape:", basket.shape)
print(basket.head())

frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)
print("Number of frequent itemsets:", len(frequent_itemsets))
print(frequent_itemsets.head(10))

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False)
print("Number of rules found:", len(rules))
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

strong_rules = rules[(rules["confidence"] >= 0.5) & (rules["lift"] >= 2.0)]
strong_rules = strong_rules[["antecedents", "consequents", "support", "confidence", "lift"]]
print("Strong rules (confidence ≥ 0.5, lift ≥ 2):")
print(strong_rules.head(10))

Basket shape: (18536, 3877)
Description   4 PURPLE FLOCK DINNER CANDLES   50'S CHRISTMAS GIFT BAG LARGE  \
InvoiceNo                                                                     
536365                                    0                               0   
536366                                    0                               0   
536367                                    0                               0   
536368                                    0                               0   
536369                                    0                               0   

Description   DOLLY GIRL BEAKER   I LOVE LONDON MINI BACKPACK  \
InvoiceNo                                                       
536365                        0                             0   
536366                        0                             0   
536367                        0                             0   
536368                        0                             0   
536369                      

## Conclusion
- Apriori found frequent itemsets by pruning product combinations that didn't meet the min_support threshold
- Association rules were generated from those itemsets and ranked by lift (strength of genuine association)
- Support: how often a combination appears across all transactions
- Confidence: how often the rule holds (given antecedent, how likely is consequent)
- Lift > 1: buying the antecedent genuinely increases the chance of buying the consequent
- Strong rules (high confidence + high lift) are directly actionable — product recommendations, bundling deals, shelf placement